In [165]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

# Import dataset

In [166]:
DATA_PATH = "../data/outputs/Walmart_Store_sales_ml_output.csv"
df = pd.read_csv(DATA_PATH)

In [167]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
0,6,2011-02-18,1572117.54,0,59.61,3.045,214.777523,6.858,2011.0,2.0,1.0,7.0,0,1
1,13,2011-03-25,1807545.43,0,42.38,3.435,128.616064,7.470,2011.0,3.0,1.0,12.0,0,0
2,17,2012-07-27,NaN,0,NaN,NaN,130.719581,5.936,2012.0,7.0,3.0,30.0,0,0
3,11,NaN,1244390.03,0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN,0,0
4,6,2010-05-28,1644470.66,0,78.89,2.759,212.412888,7.092,2010.0,5.0,2.0,21.0,0,0


In [168]:
df.shape

(150, 14)

In [169]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Store                 150 non-null    int64  
 1   Date                  132 non-null    object 
 2   Weekly_Sales          136 non-null    float64
 3   Holiday_Flag          150 non-null    int64  
 4   Temperature           132 non-null    float64
 5   Fuel_Price            136 non-null    float64
 6   CPI                   138 non-null    float64
 7   Unemployment          135 non-null    float64
 8   Year                  132 non-null    float64
 9   Month                 132 non-null    float64
 10  Quarter               132 non-null    float64
 11  Week                  132 non-null    float64
 12  Is_Year_End           150 non-null    int64  
 13  Holiday_Flag_missing  150 non-null    int64  
dtypes: float64(9), int64(4), object(1)
memory usage: 16.5+ KB


In [170]:
df.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
count,150.000000,132,1.360000e+02,150.000000,132.000000,136.000000,138.000000,135.000000,132.000000,132.000000,132.000000,132.000000,150.000000,150.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2012-10-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.073333,61.398106,3.320853,179.898509,7.598430,2010.856061,6.393939,2.454545,25.681818,0.066667,0.080000
std,6.231191,NaN,6.474630e+05,0.261556,18.378901,0.478149,40.274956,1.577173,0.811488,3.214370,1.058327,14.001809,0.250279,0.272202
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000,2010.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500,2010.000000,4.000000,2.000000,15.000000,0.000000,0.000000
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000,2011.000000,6.000000,2.000000,25.000000,0.000000,0.000000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000,2012.000000,9.000000,3.000000,36.250000,0.000000,0.000000


# Preprocessing

In [171]:
# Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Drop des lignes avec target manquante
df = df.dropna(subset=["Weekly_Sales"]).copy()

In [172]:
# Feature engineering Date
df['Year'] = df['Date'].dt.year.astype("Int64")
df['Month'] = df['Date'].dt.month.astype("Int64")
df["Quarter"] = df["Date"].dt.quarter.astype("Int64")
df['Week'] = df['Date'].dt.isocalendar().week.astype("Int64")
df["Is_Year_End"] = (df["Date"].dt.month == 12).astype(int)

# Suppression de la date
df = df.drop(columns=["Date"])

In [173]:
# Outliers sur Temperature, Fuel_Price, CPI, Unemployment
outlier_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
outlier_cols = [c for c in outlier_cols if c in df.columns]

for c in outlier_cols:
    mu, sigma = df[c].mean(), df[c].std()
    lo, hi = mu - 3*sigma, mu + 3*sigma
    df = df[(df[c] >= lo) & (df[c] <= hi)]

In [174]:
df.shape

(90, 13)

In [175]:
df.head()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
0,6,1572117.54,0,59.61,3.045,214.777523,6.858,2011,2,1,7,0,1
1,13,1807545.43,0,42.38,3.435,128.616064,7.470,2011,3,1,12,0,0
4,6,1644470.66,0,78.89,2.759,212.412888,7.092,2010,5,2,21,0,0
6,15,695396.19,0,69.80,4.069,134.855161,7.658,2011,6,2,22,0,0
7,20,2203523.20,0,39.93,3.617,213.023622,6.961,2012,2,1,5,0,0


# Pipeline

In [176]:
y = df["Weekly_Sales"]
X = df.drop(columns=["Weekly_Sales"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_cols = [c for c in ["Store", "Holiday_Flag"] if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", LinearRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment',
                                                   'Year', 'Month', 'Quarter',
                                                   'Week', 'Is_Year_End',
                                                   'Holiday_Flag_missing']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Store', 'Holiday_Flag'])])),
                ('reg', LinearRegression())])

## Baseline

### Métrique RMSE (Root Mean Squared Error)
- Il répond à la question : "Combien je me trompe, en unités métier ?"
- Le RMSE mesure l’écart moyen entre la valeur réelle y et la prédiction exprimé dans la même unité que la cible (ici, des ventes, target : Weekly_Sales).
- Plus le RMSE est faible, plus les prédictions sont proches des valeurs réelles
- Etant donné que l’erreur est mise au carré avant d’être moyennée, le RMSE pénalise davantage les grosses erreurs, ce qui est pertinent ici car rater les pics (périodes de fêtes, promos) coûte plus cher opérationnellement


### Métrique R² (coefficient de détermination)
- Il répond à la question : “Quelle part de la variabilité le modèle explique ?”
- Le R² mesure la qualité d’explication du modèle : il compare le modèle à une baseline très simple qui prédirait toujours la moyenne de y
- R² = 0.90 signifie : le modèle explique environ 90% de la variabilité observée (sur le jeu considéré). Plus c’est proche de 1, mieux c’est. Si c’est proche de 0, le modèle n’explique pas mieux qu’une prédiction constante (moyenne).

In [177]:
pred_train = model.predict(X_train)
pred_test  = model.predict(X_test)

rmse_train = root_mean_squared_error(y_train, pred_train)
rmse_test  = root_mean_squared_error(y_test,  pred_test)

r2_train = r2_score(y_train, pred_train)
r2_test  = r2_score(y_test,  pred_test)

print(f"RMSE train: {rmse_train:,.0f} | RMSE test: {rmse_test:,.0f}")
print(f"R²   train: {r2_train:.3f} | R²   test: {r2_test:.3f}")

RMSE train: 66,360 | RMSE test: 182,176
R²   train: 0.990 | R²   test: 0.900


### Interprétation des résultats

Performance globale : 
- R² test = 0.90 est élevé : Le modèle capte une grande partie du signal
- Mais le RMSE test est nettement plus haut : en moyenne, le modèle se trompe d’environ 182k unités de ventes hebdomadaires sur les observations de test

Cas d'overfitting - Généralisation (train vs test) :
- RMSE train = 66k vs RMSE test = 182k : l’erreur explose sur le test (environ ×2,7).
- R² train = 0.99 vs R² test = 0.90 : le modèle colle presque parfaitement au train, mais perd en généralisation. Donc le modèle apprend très bien les particularités du train (bruit, spécificités) mais il généralise moins bien

In [178]:
df_pred = pd.DataFrame({"y_true": y_test, "y_pred": pred_test})
fig = px.scatter(df_pred, x="y_true", y="y_pred", title="Test : y_true vs y_pred", opacity=0.6)
fig.add_shape(type="line",
              x0=df_pred["y_true"].min(), y0=df_pred["y_true"].min(),
              x1=df_pred["y_true"].max(), y1=df_pred["y_true"].max())
fig.show()

In [179]:
# Récupération des noms de features
ct = model.named_steps["preprocess"]
feature_names = ct.get_feature_names_out()

coefs = model.named_steps["reg"].coef_
coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()

display(coef_df.sort_values("abs_coef", ascending=False).head(20))

,feature,coef,abs_coef
13,cat__Store_4,2.511710e+06,2.511710e+06
21,cat__Store_13,2.329861e+06,2.329861e+06
19,cat__Store_10,2.287352e+06,2.287352e+06
12,cat__Store_3,-2.108523e+06,2.108523e+06
14,cat__Store_5,-2.042073e+06,2.042073e+06
18,cat__Store_9,-1.949762e+06,1.949762e+06
27,cat__Store_19,1.661883e+06,1.661883e+06
17,cat__Store_8,-1.521141e+06,1.521141e+06
26,cat__Store_18,1.326033e+06,1.326033e+06
2,num__CPI,1.222218e+06,1.222218e+06


In [180]:
est = model

ct = est.named_steps["preprocess"]
names = ct.get_feature_names_out()
coefs = est.named_steps["reg"].coef_

coef_df = pd.DataFrame({"feature": names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()

top = (
    coef_df.sort_values("abs_coef", ascending=False)
           .head(20)
           .sort_values("coef")
)

fig = px.bar(
    top,
    x="coef",
    y="feature",
    orientation="h",
    title="Baseline (LinearRegression) : Top 20 coefficients (valeur absolue)",
    labels={"coef": "Coefficient", "feature": "Feature"}
)
fig.show()

## Modèle régularisé (Ridge + GridSearchCV)

In [181]:
ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", Ridge())
])

param_grid = {
    "reg__alpha": np.logspace(-3, 4, 20)
}

gs_ridge = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gs_ridge.fit(X_train, y_train)

best_ridge = gs_ridge.best_estimator_
pred_train = best_ridge.predict(X_train)
pred_test  = best_ridge.predict(X_test)

print("Best alpha:", gs_ridge.best_params_["reg__alpha"])
print(f"RMSE train: {root_mean_squared_error(y_train, pred_train):,.0f} | RMSE test: {root_mean_squared_error(y_test, pred_test):,.0f}")
print(f"R² train: {r2_score(y_train, pred_train):.3f} | R² test: {r2_score(y_test, pred_test):.3f}")


Best alpha: 0.00545559478116852
RMSE train: 70,891 | RMSE test: 150,862
R² train: 0.989 | R² test: 0.931


### Interprétation des résultats

Performance globale : 
- R² test = 0.931 : le modèle explique une très grande partie de la variabilité des ventes hebdomadaires, et fait mieux que la baseline (0.90).
- RMSE test = 150,862 : l’erreur moyenne sur le jeu de test baisse nettement par rapport à la baseline (de 182k à 151k, soit -17%), ce qui traduit une amélioration concrète en unités métier.

Généralisation (train vs test) : réduction de l’overfitting
- RMSE train = 70,891 vs RMSE test = 150,862 : l’écart train/test reste présent, mais il est moins problématique car le test s’améliore fortement (l’objectif principal).
- R² train = 0.989 vs R² test = 0.931 : le modèle reste très performant sur train, tout en gagnant en robustesse sur des données non vues.

In [183]:
pred_test_ridge = best_ridge.predict(X_test)

df_pred = pd.DataFrame({
    "y_true": y_test,
    "y_pred": pred_test_ridge
})

fig = px.scatter(
    df_pred,
    x="y_true",
    y="y_pred",
    title="Ridge (test) : y_true vs y_pred",
    opacity=0.6
)

# Diagonale parfaite y = x
m = df_pred["y_true"].min()
M = df_pred["y_true"].max()
fig.add_shape(type="line", x0=m, y0=m, x1=M, y1=M)

fig.show()

In [184]:
res = pd.DataFrame(gs_ridge.cv_results_)
res["rmse_cv"] = -res["mean_test_score"]

fig = px.line(
    res,
    x="param_reg__alpha",
    y="rmse_cv",
    markers=True,
    title="Ridge : RMSE CV selon alpha",
    labels={"param_reg__alpha":"alpha", "rmse_cv":"RMSE (CV)"}
)
fig.update_xaxes(type="log")
fig.show()

In [185]:
# Récupération des noms de features
best_ridge = gs_ridge.best_estimator_
ct = best_ridge.named_steps["preprocess"]

feature_names = ct.get_feature_names_out()
coefs = best_ridge.named_steps["reg"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})
coef_df["abs_coef"] = coef_df["coef"].abs()

coef_df.sort_values("abs_coef", ascending=False).head(20)

,feature,coef,abs_coef
13,cat__Store_4,1.243449e+06,1.243449e+06
14,cat__Store_5,-1.213922e+06,1.213922e+06
12,cat__Store_3,-1.182814e+06,1.182814e+06
18,cat__Store_9,-1.076069e+06,1.076069e+06
21,cat__Store_13,1.066270e+06,1.066270e+06
19,cat__Store_10,9.937120e+05,9.937120e+05
24,cat__Store_16,-8.224991e+05,8.224991e+05
22,cat__Store_14,8.222615e+05,8.222615e+05
16,cat__Store_7,-7.222171e+05,7.222171e+05
17,cat__Store_8,-6.467069e+05,6.467069e+05


In [186]:
top = coef_df.sort_values("abs_coef", ascending=False).head(20).sort_values("coef")

fig = px.bar(
    top,
    x="coef",
    y="feature",
    orientation="h",
    title="Ridge : Top 20 coefficients (valeur absolue)",
    labels={"coef":"Coefficient Ridge", "feature":"Feature"}
)
fig.show()

## Modèle régularisé (Lasso + GridSearchCV)

In [187]:
lasso_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", Lasso(max_iter=20000))
])

param_grid_lasso = {
    "reg__alpha": np.logspace(-4, 2, 20)
}

gs_lasso = GridSearchCV(
    lasso_pipe,
    param_grid=param_grid_lasso,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gs_lasso.fit(X_train, y_train)

best_lasso = gs_lasso.best_estimator_
pred_train = best_lasso.predict(X_train)
pred_test  = best_lasso.predict(X_test)

print("Best alpha:", gs_lasso.best_params_["reg__alpha"])
print(f"RMSE train: {root_mean_squared_error(y_train, pred_train):,.0f} | RMSE test: {root_mean_squared_error(y_test, pred_test):,.0f}")
print(f"R² train: {r2_score(y_train, pred_train):.3f} | R² test: {r2_score(y_test, pred_test):.3f}")


Best alpha: 48.32930238571752
RMSE train: 69,878 | RMSE test: 152,926
R² train: 0.989 | R² test: 0.929


### Interprétation des résultats

Performance globale : 
- R² test = 0.929 : le modèle explique une grande partie de la variabilité des ventes hebdomadaires, nettement mieux que la baseline (0.90).
- RMSE test = 152,926 : l’erreur moyenne sur le test diminue fortement par rapport à la baseline (de 182k à 153k soit -16%), ce qui traduit un gain concret en unités métier.

Généralisation (train vs test) : réduction de l’overfitting
- RMSE train = 69,878 vs RMSE test = 152,926 : l’écart train/test reste présent, mais le niveau d’erreur sur test est nettement meilleur que la baseline, donc ca semble indiquer une meilleure robustesse globale.
- R² train = 0.989 vs R² test = 0.929 : la performance est élevée sur train et conserve un bon niveau sur test, ce qui confirme une généralisation correcte.

In [188]:
pred_test_lasso = best_lasso.predict(X_test)

df_pred = pd.DataFrame({
    "y_true": y_test,
    "y_pred": pred_test_ridge
})

fig = px.scatter(
    df_pred,
    x="y_true",
    y="y_pred",
    title="Ridge (test) : y_true vs y_pred",
    opacity=0.6
)

# Diagonale parfaite y = x
m = df_pred["y_true"].min()
M = df_pred["y_true"].max()
fig.add_shape(type="line", x0=m, y0=m, x1=M, y1=M)

fig.show()

In [189]:
# Récupération des noms de features
best_lasso = gs_lasso.best_estimator_

ct = best_lasso.named_steps["preprocess"]
names = ct.get_feature_names_out()
coefs = best_lasso.named_steps["reg"].coef_

coef_lasso = pd.DataFrame({"feature": names, "coef": coefs})
n_zero = (coef_lasso["coef"] == 0).sum()
print("Coefficients à 0:", n_zero, "/", len(coef_lasso))

display(coef_lasso.loc[coef_lasso["coef"] != 0].assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False).head(20))

Coefficients à 0: 1 / 31


,feature,coef,abs_coef
14,cat__Store_5,-1.382503e+06,1.382503e+06
12,cat__Store_3,-1.362033e+06,1.362033e+06
13,cat__Store_4,1.336547e+06,1.336547e+06
18,cat__Store_9,-1.249580e+06,1.249580e+06
21,cat__Store_13,1.158128e+06,1.158128e+06
19,cat__Store_10,1.090340e+06,1.090340e+06
24,cat__Store_16,-9.211284e+05,9.211284e+05
16,cat__Store_7,-8.251538e+05,8.251538e+05
17,cat__Store_8,-8.204273e+05,8.204273e+05
22,cat__Store_14,7.427549e+05,7.427549e+05


In [190]:
est = best_lasso

ct = est.named_steps["preprocess"]
feature_names = ct.get_feature_names_out()
coefs = est.named_steps["reg"].coef_

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()

top = (
    coef_df.sort_values("abs_coef", ascending=False)
           .head(20)
           .sort_values("coef")
)

fig = px.bar(
    top,
    x="coef",
    y="feature",
    orientation="h",
    title="Lasso : Top 20 coefficients (valeur absolue)",
    labels={"coef": "Coefficient Lasso", "feature": "Feature"},
)
fig.show()